# 01 — Análisis Exploratorio del Dataset Bioacústico

**Proyecto:** Sistema de Identificación de Fauna por Bioacústica + IA  
**Autor:** Ian  
**Objetivo:** Caracterizar el dataset multitaxonómico antes del entrenamiento del modelo.

## Contenido
1. Carga y validación del dataset
2. Distribución de clases y balance
3. Estadísticas de duración de grabaciones
4. Visualización de espectrogramas por grupo taxonómico
5. Análisis de características acústicas (MFCC, Mel)
6. Correlaciones entre features espectrales
7. Detección de outliers acústicos
8. Estimación del poder discriminativo por feature

In [ ]:
# ── Dependencias ────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '..')  # acceso a src/

import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DATA_DIR = Path('../data/raw')
print(f'Directorio de datos: {DATA_DIR.resolve()}')
print(f'Existe: {DATA_DIR.exists()}')

## 1. Carga y Validación del Dataset

In [ ]:
# ── Inventario de archivos ───────────────────────────────────────────────────
records = []
AUDIO_EXTS = {'.wav', '.mp3', '.flac', '.ogg'}

for class_dir in sorted(DATA_DIR.iterdir()):
    if not class_dir.is_dir():
        continue
    for fp in class_dir.iterdir():
        if fp.suffix.lower() not in AUDIO_EXTS:
            continue
        # Leer metadata JSON si existe
        meta = {}
        meta_path = fp.with_suffix('.json')
        if meta_path.exists():
            try:
                meta = json.loads(meta_path.read_text())
            except Exception:
                pass
        # Info de audio
        try:
            info = sf.info(str(fp))
            records.append({
                'filepath':     str(fp),
                'class':        class_dir.name,
                'filename':     fp.name,
                'duration_s':   info.duration,
                'sample_rate':  info.samplerate,
                'channels':     info.channels,
                'format':       info.format,
                'source':       meta.get('source', 'unknown'),
                'quality':      meta.get('quality', ''),
                'country':      meta.get('country', ''),
                'lat':          meta.get('lat', None),
                'lng':          meta.get('lng', None),
            })
        except Exception as e:
            print(f'  ERROR {fp.name}: {e}')

df = pd.DataFrame(records)
print(f'Total archivos válidos: {len(df)}')
print(f'Clases encontradas:     {df["class"].nunique()}')
print(f'\nTipos de formato:\n{df["format"].value_counts()}')
df.head()

## 2. Distribución de Clases

In [ ]:
# ── Conteo por clase ─────────────────────────────────────────────────────────
class_counts = df['class'].value_counts().reset_index()
class_counts.columns = ['class', 'count']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras horizontales
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(class_counts)))
bars = ax.barh(class_counts['class'], class_counts['count'], color=colors)
ax.set_xlabel('Número de grabaciones')
ax.set_title('Distribución de muestras por clase', fontweight='bold')
ax.axvline(class_counts['count'].mean(), color='red', linestyle='--', alpha=0.7, label=f'Media={class_counts["count"].mean():.0f}')
ax.legend()
for bar, val in zip(bars, class_counts['count']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)

# Pie por fuente
ax2 = axes[1]
src_counts = df['source'].value_counts()
ax2.pie(src_counts.values, labels=src_counts.index, autopct='%1.1f%%',
        startangle=90, colors=plt.cm.Set3(np.linspace(0, 1, len(src_counts))))
ax2.set_title('Distribución por fuente de datos', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/visualizations/class_distribution.png', bbox_inches='tight')
plt.show()

print('\nEstadísticas de balance:')
print(f'  Clase con más muestras:  {class_counts.iloc[0]["class"]} ({class_counts.iloc[0]["count"]})')
print(f'  Clase con menos muestras:{class_counts.iloc[-1]["class"]} ({class_counts.iloc[-1]["count"]})')
print(f'  Ratio max/min:           {class_counts["count"].max() / class_counts["count"].min():.2f}x')
print(f'  Imbalance severo (>5x):  {(class_counts["count"].max() / class_counts["count"].min()) > 5}')

## 3. Análisis de Duración de Grabaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma de duración global
ax = axes[0, 0]
ax.hist(df['duration_s'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(df['duration_s'].median(), color='red', linestyle='--', label=f'Mediana={df["duration_s"].median():.1f}s')
ax.set_xlabel('Duración (segundos)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución global de duración')
ax.legend()

# Box plot por clase
ax2 = axes[0, 1]
top_classes = class_counts['class'].head(15).tolist()
df_top = df[df['class'].isin(top_classes)]
df_top.boxplot(column='duration_s', by='class', ax=ax2, rot=45)
ax2.set_title('Duración por clase (top 15)')
ax2.set_xlabel('')
plt.sca(ax2)
plt.xticks(fontsize=8)

# Distribución de sample rate
ax3 = axes[1, 0]
sr_counts = df['sample_rate'].value_counts()
ax3.bar(sr_counts.index.astype(str), sr_counts.values, color='coral')
ax3.set_xlabel('Sample rate (Hz)')
ax3.set_ylabel('N° grabaciones')
ax3.set_title('Distribución de tasas de muestreo')

# Duración total por clase
ax4 = axes[1, 1]
total_dur = df.groupby('class')['duration_s'].sum().sort_values(ascending=True).tail(15)
total_dur.plot(kind='barh', ax=ax4, color='mediumpurple')
ax4.set_xlabel('Duración total (segundos)')
ax4.set_title('Audio total por clase (top 15)')

plt.suptitle('Análisis de Duración del Dataset Bioacústico', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/visualizations/duration_analysis.png', bbox_inches='tight')
plt.show()

print('\nEstadísticas globales de duración:')
print(df['duration_s'].describe().round(2))
print(f'\nDuración total del dataset: {df["duration_s"].sum() / 3600:.2f} horas')

## 4. Visualización de Espectrogramas por Grupo Taxonómico

In [ ]:
def plot_spectrogram_panel(filepath: str, class_name: str, ax_wave, ax_mel, ax_mfcc,
                            sr_target: int = 22050, duration: float = 3.0):
    """Dibuja waveform + Mel spectrogram + MFCC para un archivo de audio."""
    y, sr = librosa.load(filepath, sr=sr_target, duration=duration, mono=True)
    
    # Waveform
    times = np.linspace(0, len(y)/sr, len(y))
    ax_wave.plot(times, y, color='steelblue', linewidth=0.5, alpha=0.8)
    ax_wave.set_ylabel('Amplitud')
    ax_wave.set_title(f'{class_name}', fontweight='bold', fontsize=10)
    ax_wave.set_xlim([0, duration])
    
    # Mel spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, x_axis='time', y_axis='mel',
                                    sr=sr, fmax=8000, ax=ax_mel, cmap='magma')
    ax_mel.set_ylabel('Mel (Hz)')
    ax_mel.set_xlabel('')
    
    # MFCC (primeros 20)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    librosa.display.specshow(mfcc, x_axis='time', ax=ax_mfcc, cmap='coolwarm')
    ax_mfcc.set_ylabel('MFCC')
    ax_mfcc.set_xlabel('Tiempo (s)')

# Seleccionar 1 archivo por clase (máximo 6 clases)
sample_files = []
for cls in df['class'].unique()[:6]:
    row = df[df['class'] == cls].sample(1).iloc[0]
    sample_files.append((row['filepath'], cls))

n = len(sample_files)
fig = plt.figure(figsize=(20, n * 4))
gs  = gridspec.GridSpec(n, 3, figure=fig, hspace=0.6, wspace=0.3)

for i, (fp, cls) in enumerate(sample_files):
    try:
        ax1 = fig.add_subplot(gs[i, 0])
        ax2 = fig.add_subplot(gs[i, 1])
        ax3 = fig.add_subplot(gs[i, 2])
        plot_spectrogram_panel(fp, cls, ax1, ax2, ax3)
    except Exception as e:
        print(f'Error con {cls}: {e}')

fig.suptitle('Espectrogramas Mel y MFCC por Clase Taxonómica', 
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('../results/visualizations/spectrograms_panel.png', bbox_inches='tight')
plt.show()

## 5. Extracción Masiva de Características Acústicas

In [ ]:
from tqdm.notebook import tqdm
import sys
sys.path.insert(0, '..')
from src.audio_processing.preprocessor import AudioPreprocessor, AudioConfig

proc = AudioPreprocessor(AudioConfig(sample_rate=22050))
feat_records = []

# Muestra representativa para EDA (máximo 50 por clase)
df_sample = df.groupby('class').apply(lambda x: x.sample(min(50, len(x)))).reset_index(drop=True)
print(f'Extrayendo features de {len(df_sample)} archivos...')

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    try:
        y, sr = librosa.load(row['filepath'], sr=22050, duration=3.0, mono=True)
        feats  = proc.spectral_features(y)
        mfcc   = proc.mfcc(y, include_delta=False)  # (40, T)
        
        # Estadísticas MFCC
        mfcc_mean = mfcc.mean(axis=1)  # (40,)
        
        entry = {
            'class':      row['class'],
            'filepath':   row['filepath'],
            'source':     row['source'],
            **feats,
        }
        for j, m in enumerate(mfcc_mean):
            entry[f'mfcc_{j+1}'] = float(m)
        
        feat_records.append(entry)
    except Exception as e:
        pass  # Skip archivos problemáticos

df_feats = pd.DataFrame(feat_records)
print(f'Features extraídas: {len(df_feats)} muestras × {df_feats.shape[1]} columnas')
df_feats.head(3)

## 6. Visualización de Features por Clase

In [ ]:
# ── Violin plots de features espectrales clave ───────────────────────────────
spec_features = ['spectral_centroid_mean', 'spectral_bandwidth_mean',
                 'zcr_mean', 'rms_mean', 'spectral_flatness_mean']

fig, axes = plt.subplots(1, len(spec_features), figsize=(20, 5))

for ax, feat in zip(axes, spec_features):
    if feat not in df_feats.columns:
        continue
    classes_present = df_feats['class'].unique()
    data = [df_feats[df_feats['class'] == c][feat].dropna().values
            for c in classes_present]
    data = [d for d in data if len(d) > 0]
    
    parts = ax.violinplot(data, showmedians=True)
    ax.set_xticks(range(1, len(classes_present)+1))
    ax.set_xticklabels([c.replace('_', '\n') for c in classes_present],
                        fontsize=7, rotation=45)
    ax.set_title(feat.replace('_mean','').replace('_',' '), fontsize=9)
    for pc in parts['bodies']:
        pc.set_alpha(0.7)

plt.suptitle('Distribución de Features Espectrales por Clase', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/visualizations/spectral_features_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Heatmap de MFCCs medios por clase ────────────────────────────────────────
mfcc_cols = [c for c in df_feats.columns if c.startswith('mfcc_')]

if mfcc_cols:
    mfcc_by_class = df_feats.groupby('class')[mfcc_cols].mean()
    
    fig, ax = plt.subplots(figsize=(16, max(4, len(mfcc_by_class) * 0.5)))
    sns.heatmap(
        mfcc_by_class,
        cmap='RdBu_r',
        center=0,
        xticklabels=[f'MFCC {i}' for i in range(1, len(mfcc_cols)+1)],
        yticklabels=mfcc_by_class.index,
        ax=ax, linewidths=0.3,
    )
    ax.set_title('MFCC medios por clase taxonómica', fontweight='bold', fontsize=12)
    ax.set_xlabel('Coeficiente MFCC')
    ax.set_ylabel('Clase')
    plt.tight_layout()
    plt.savefig('../results/visualizations/mfcc_heatmap.png', bbox_inches='tight')
    plt.show()

## 7. Reducción Dimensional — PCA y t-SNE

In [ ]:
# ── Preparar matriz de features ──────────────────────────────────────────────
feature_cols = [c for c in df_feats.columns
                if c not in ['class', 'filepath', 'source']
                and df_feats[c].dtype in ['float64', 'float32']]

X = df_feats[feature_cols].fillna(0).values
y_labels = df_feats['class'].values

# Normalizar
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Varianza explicada (PC1-PC2): {pca.explained_variance_ratio_[:2].sum()*100:.1f}%')
print(f'Varianza explicada (PC1-PC10): {pca.explained_variance_ratio_[:10].sum()*100:.1f}%')

# t-SNE sobre los primeros 50 componentes PCA
if len(X_pca) > 30:  # t-SNE requiere muestras suficientes
    tsne = TSNE(n_components=2, perplexity=min(30, len(X_pca)//4), 
                random_state=42, n_iter=1000)
    X_tsne = tsne.fit_transform(X_pca)
    print('t-SNE completado')
else:
    X_tsne = X_pca[:, :2]
    print('Pocas muestras — usando PCA en lugar de t-SNE')

In [ ]:
# ── Visualización PCA + t-SNE ────────────────────────────────────────────────
unique_classes = sorted(set(y_labels))
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_classes)))
cmap   = {c: colors[i] for i, c in enumerate(unique_classes)}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (X_red, title) in zip(axes, [
    (X_pca[:, :2], f'PCA — PC1 vs PC2 ({pca.explained_variance_ratio_[:2].sum()*100:.1f}% var.)'),
    (X_tsne,       't-SNE (perplexity=30)'),
]):
    for cls in unique_classes:
        mask = y_labels == cls
        ax.scatter(X_red[mask, 0], X_red[mask, 1],
                   c=[cmap[cls]], label=cls.replace('_', ' '),
                   alpha=0.65, s=30, edgecolors='none')
    ax.set_title(title, fontweight='bold')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8,
              markerscale=1.5, framealpha=0.8)
    ax.set_xlabel('Dimensión 1')
    ax.set_ylabel('Dimensión 2')

plt.suptitle('Estructura del Espacio de Features Bioacústicos', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/visualizations/pca_tsne.png', bbox_inches='tight')
plt.show()

## 8. Poder Discriminativo por Feature — ANOVA F-score

In [ ]:
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y_labels)

F_scores, p_values = f_classif(X_scaled, y_enc)

feat_importance = pd.DataFrame({
    'feature':  feature_cols,
    'F_score':  F_scores,
    'p_value':  p_values,
    'significant': p_values < 0.01,
}).sort_values('F_score', ascending=False)

# Top 25 features
top25 = feat_importance.head(25)

fig, ax = plt.subplots(figsize=(10, 8))
colors_bar = ['#2ecc71' if s else '#e74c3c' for s in top25['significant']]
bars = ax.barh(range(len(top25)), top25['F_score'], color=colors_bar, alpha=0.8)
ax.set_yticks(range(len(top25)))
ax.set_yticklabels(top25['feature'].str.replace('_', ' '), fontsize=9)
ax.set_xlabel('F-score (ANOVA)')
ax.set_title('Top 25 Features por Poder Discriminativo\n(verde = p<0.01)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/visualizations/feature_importance.png', bbox_inches='tight')
plt.show()

print(f'\nFeatures significativas (p<0.01): {feat_importance["significant"].sum()}/{len(feat_importance)}')
print('\nTop 10 features más discriminativas:')
print(feat_importance.head(10)[['feature','F_score','p_value']].to_string(index=False))

## 9. Resumen para Tesis

In [ ]:
# ── Tabla resumen del dataset ─────────────────────────────────────────────────
summary = df.groupby('class').agg(
    n_files         = ('filepath', 'count'),
    total_dur_min   = ('duration_s', lambda x: round(x.sum()/60, 1)),
    mean_dur_s      = ('duration_s', lambda x: round(x.mean(), 1)),
    median_dur_s    = ('duration_s', lambda x: round(x.median(), 1)),
    primary_source  = ('source',    lambda x: x.mode()[0] if len(x) else ''),
    sample_rates    = ('sample_rate', lambda x: ', '.join(map(str, sorted(x.unique())))),
).reset_index()

print('='*80)
print('RESUMEN DEL DATASET — BioAcoustics MultiTaxa v1')
print('='*80)
print(summary.to_string(index=False))
print('='*80)
print(f'TOTAL FILES:     {df["filepath"].count()}')
print(f'TOTAL DURATION:  {df["duration_s"].sum()/3600:.2f} hours')
print(f'TOTAL CLASSES:   {df["class"].nunique()}')
print(f'SOURCES:         {", ".join(df["source"].unique())}')

# Guardar resumen CSV
summary.to_csv('../results/reports/dataset_summary.csv', index=False)
print('\nGuardado: results/reports/dataset_summary.csv')